In [ ]:
from functools import partial
from typing import List, Tuple, Union

import numpy as np
from ipywidgets import widgets, HBox, VBox, Layout
from IPython.display import display

In [ ]:
X = +1
O = -1
EMPTY_SPACE = 0

INTERFACE_MAPPING = {
    X: 'X',
    O: 'O'
}

In [ ]:
class Interface:
    def __init__(self):
        self.buttons = [
            [
                widgets.Button(
                    description='',
                    layout=Layout(width='100px', height='100px')
                )
                for _ in range(3)
            ]
            for _ in range(3)
        ]

        self.board_widget = VBox([HBox(row) for row in self.buttons])

        self.__player = X
        self.link_positions()

    def on_select_position(self, pos: tuple, button: widgets.Button):
        """Callback on user click on a position of the board

        :param pos: row, column of button on grid
        :param button: clicked button
        """
        if not button.description:
            button.description = INTERFACE_MAPPING.get(self.__player, '')
            self.__player *= -1  # invert player

    def link_positions(self):
        """Link clicks on buttons"""
        for i, row in enumerate(self.buttons):
            for j, button in enumerate(row):
                button.on_click(partial(self.on_select_position, (i, j)))

    def disable_buttons(self):
        """Disable buttons to avoid clicks after end game"""
        for row in self.buttons:
            for button in row:
                button.disabled = True


    def start(self):
        """Display board interface"""
        display(self.board_widget)

In [ ]:
board = np.zeros((3, 3), dtype=int)
board[:, 1] = O
np.diag(board)
board

In [ ]:
Sua vez, crie uma função para verificar se o jogo acabou, quais são as próximas possibilidades de jogadas e obter um score pro jogo
Pseudocode:

receba uma board

somas_finalizadoras = [3, -3]

para cada linha:
    verifica se a soma é uma somas_finalizadoras

para cada coluna:
    verifica se a soma é uma somas_finalizadoras
    
verifica se a soma da diagonal principal é finalizadora

verifica se a soma da diagonal secundária é uma soma finalizadora

verifica se há posições vazias

In [ ]:
def get_game_status(tabuleiro: np.ndarray) -> Tuple[bool, Union[int, None]]:
    # se tem 3 X ou 3 O em uma linha
    for linha in tabuleiro:
        if sum(linha) == 3 * X:
            return True, X
        elif sum(linha) == 3 * O:
            return True, O

    # se tem 3 X ou 3 O em uma coluna
    for col in tabuleiro.T:
        if sum(col) == 3 * X:
            return True, X
        elif sum(col) == 3 * O:
            return True, O

    # diagonal
    diagonal = sum(np.diag(tabuleiro))
    if diagonal == 3 * X:
        return True, X
    elif diagonal == 3 * 0:
        return True, O

    # segunda diagonal
    segunda_diagonal = sum(np.diagonal(np.fliplr(board)))
    if segunda_diagonal == 3 * X:
            return True, X
    elif segunda_diagonal == 3 * 0:
        return True, O

    # verifica empate
    empate = np.all(tabuleiro != EMPTY_SPACE)
    return empate, None

In [ ]:
def get_possible_moves(tabuleiro: np.ndarray, jogador: int = X) -> List[np.ndarray]:
    """Get next possible moves by some player
    """
    possiveis_jogadas = []

    for linha in range(3):
        for coluna in range(3):
            if tabuleiro[linha, coluna] == EMPTY_SPACE:
                jogada = board.copy()
                jogada[linha, coluna] = jogador
                possiveis_jogadas.append(jogada)

    return possiveis_jogadas

In [ ]:
AI_PLAYER = X
HUMAN_PLAYER = O
MAX_N_MOVES = 9

def get_score(winner: int, n_moves: int) -> int:
    """Get how well was the game for the AI:
        - win faster is better than win slower
        - lose slower is better than win faster
        - draw is a intermediary result
    """
    # Pseudocódigo:
    # receba um ganhador e a quantidade de movimentos

    # se o ganhador for a IA:
    #     retorne MAX_N_MOVES + 1 - n_movex
    if winner == AI_PLAYER:
        return MAX_N_MOVES + 1 - n_moves
    # se for o humano
    #     retorne -(MAX_N_MOVES + 1 - n_movex)
    elif winner == HUMAN_PLAYER:
        return -(MAX_N_MOVES + 1 - n_moves)
    # senão
    #     retorna 0
    else:
        return 0

In [ ]:
# @markdown ## Testando fução de score

winner_param_value = -1 # @param ["AI_PLAYER", "HUMAN_PLAYER", "None"] {allow-input: true}

# If the parameter value is a string, evaluate it. Otherwise, use it directly.
if isinstance(winner_param_value, str):
    winner = eval(winner_param_value)
else:
    winner = winner_param_value

n_moves = 0 # @param {type:"slider", min:0, max:9, step:1}

print(get_score(winner, n_moves))

In [ ]:
def mini_max(board, player=AI_PLAYER, n_moves=0):
    # Check if it is a maximization or minimization step (who is the player?)
    # Get game status and if it over
    is_over, winner = get_game_status(board)
    # if game over return the score and board
    if is_over:
        return get_score(winner, n_moves), board
    # othewise, for each possible move call mini_max and get the score and board
    next_player = X if player == O else O
    scores_and_moves = []
    for move in get_possible_moves(board, player):
        score, _ = mini_max(move, next_player, n_moves + 1)
        scores_and_moves.append((score, move))

    # choose the action to maximize the score if it is a maximization step
    if player == AI_PLAYER:
        return max(scores_and_moves, key=lambda x: x[0])
    # otherwise, choose the action to minimize the score
    else:
        return min(scores_and_moves, key=lambda x: x[0])

In [ ]:
board = np.zeros(shape=(3, 3), dtype=np.int8)
_, new_board = mini_max(board, AI_PLAYER)
new_board

In [ ]:
new_board = new_board.copy()
new_board[1, 0] = -1
_, new_board = mini_max(new_board, AI_PLAYER)
new_board

In [ ]:
game = TicTacToeAI(ai_starts=False)
game.start()